# 02 – Distance-Only Benchmark & Ridge Regression (Member 2)

**Project:** Detecting and predicting earthquake ground-motion directionality patterns  
**Target:** Δθ = |θᵢ − θⱼ| – absolute difference in the direction of maximum shaking between two recording sites of the same earthquake.

**This notebook covers**
1. Loading Member 1's processed *pair* dataset
2. Creating the shared train / validation / test split (saved so Member 3 reuses it)
3. Distance-only benchmark
4. Ridge Regression with α tuning
5. Metrics (R², MAE, RMSE) + plots, saved to `results/`

## 1. Setup & configuration

In [ ]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 5)

# ----------------------------- CONFIG (edit to match Member 1's file) -----------------------------
PAIRS_CSV  = "dataset/pairs_dataset.csv"   # Member 1's processed site-pair dataset
TARGET     = "delta_theta"                 # |theta_i - theta_j|  (degrees)
DIST_COL   = "site_distance_km"            # distance between the two sites
GROUP_COL  = "EQID"                        # earthquake id -> keeps one quake entirely in one split
DROP_COLS  = ["EQID", "Earthquake Name", "Station Name", "Station Name_1", "Station Name_2",
              "Record Sequence Number", "RSN_1", "RSN_2"]   # identifiers, never used as features
SEED       = 42
TEST_SIZE, VAL_SIZE = 0.15, 0.15           # 70 / 15 / 15
SPLIT_CSV  = "dataset/split_indices.csv"   # shared with Member 3
os.makedirs("results/figures", exist_ok=True)

## 2. Load the processed pair dataset

In [ ]:
if not os.path.exists(PAIRS_CSV):
    raise FileNotFoundError(
        f"{PAIRS_CSV} not found. Member 1 must push the paired dataset with the target "
        f"'{TARGET}' and '{DIST_COL}' first (see README / hand-off note).")
df = pd.read_csv(PAIRS_CSV)
print("Shape:", df.shape)
assert TARGET in df.columns and DIST_COL in df.columns and GROUP_COL in df.columns, "check CONFIG column names"
print("Missing values:", int(df.isna().sum().sum()))
df[[TARGET, DIST_COL]].describe()

Δθ should lie in **[0°, 90°]** if orientations are measured modulo 180° (a maximum-shaking direction and its opposite are the same axis), or [0°, 180°] otherwise. We check which, since it determines what a "no-skill" prediction looks like.

In [ ]:
print("Target range:", df[TARGET].min(), "→", df[TARGET].max())
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[TARGET], bins=36, kde=True, ax=ax[0]); ax[0].set_title("Distribution of Δθ"); ax[0].set_xlabel("Δθ (deg)")
ax[1].scatter(df[DIST_COL], df[TARGET], s=4, alpha=.25); ax[1].set_xscale("log")
ax[1].set_xlabel("Site-to-site distance (km)"); ax[1].set_ylabel("Δθ (deg)"); ax[1].set_title("Δθ vs distance")
plt.tight_layout(); plt.savefig("results/figures/m2_target_overview.png", dpi=150); plt.show()

## 3. Feature matrix
All numeric columns except identifiers and the target are used. Categorical columns (e.g. NEHRP class) are one-hot encoded.  
**Scaling is fitted on the training set only** (inside a `Pipeline`) to avoid data leakage, as Member 1 noted.

In [ ]:
feat_df = df.drop(columns=[c for c in DROP_COLS + [TARGET] if c in df.columns and c != GROUP_COL])
groups = df[GROUP_COL].values
feat_df = feat_df.drop(columns=[GROUP_COL], errors="ignore")
cat_cols = feat_df.select_dtypes(exclude=[np.number]).columns.tolist()
feat_df = pd.get_dummies(feat_df, columns=cat_cols, drop_first=True, dtype=float)
X, y = feat_df, df[TARGET].values
print(f"{X.shape[1]} features:", list(X.columns))

## 4. Shared train / validation / test split
The split is **grouped by earthquake**: all site-pairs from one earthquake go into the same subset. Otherwise pairs from the same event (which share magnitude, mechanism, etc.) would leak between train and test and inflate scores.  
Indices are saved to `dataset/split_indices.csv` so Member 3's Neural Network uses *exactly* the same rows → fair comparison.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
trainval_idx, test_idx = next(gss.split(X, y, groups))
gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED)
tr_rel, va_rel = next(gss2.split(X.iloc[trainval_idx], y[trainval_idx], groups[trainval_idx]))
train_idx, val_idx = trainval_idx[tr_rel], trainval_idx[va_rel]

split = pd.Series("train", index=df.index); split.iloc[val_idx] = "val"; split.iloc[test_idx] = "test"
split.rename("split").to_frame().to_csv(SPLIT_CSV, index_label="row")
for n, i in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    print(f"{n:5s}: {len(i):6d} pairs, {len(np.unique(groups[i])):4d} earthquakes")
Xtr, Xva, Xte = X.iloc[train_idx], X.iloc[val_idx], X.iloc[test_idx]
ytr, yva, yte = y[train_idx], y[val_idx], y[test_idx]
gtr = groups[train_idx]

## 5. Evaluation helper
We report **R²** and **MAE** (required) plus RMSE. Also a trivial *mean-predictor* reference: any model with R² ≤ 0 is no better than always predicting the training mean.

In [ ]:
def metrics(name, split_name, y_true, y_pred):
    return dict(model=name, split=split_name, R2=r2_score(y_true, y_pred),
                MAE=mean_absolute_error(y_true, y_pred), RMSE=np.sqrt(mean_squared_error(y_true, y_pred)))
results = []
mean_pred = np.full_like(yte, ytr.mean(), dtype=float)
results.append(metrics("Mean predictor", "test", yte, mean_pred))
results[-1]

## 6. Benchmark – distance-only model
Simple baseline: Δθ is predicted from **site-to-site distance alone** with ordinary linear regression (on log-distance as well, since distance spans orders of magnitude). The better of the two (chosen on the *validation* set) is the benchmark. Directionality should decorrelate with separation, so this is a physically sensible baseline that the full-feature models must beat.

In [ ]:
def dist_feats(d, mode):
    d = np.asarray(d, float).reshape(-1, 1)
    return np.log10(d + 1e-3) if mode == "log" else d

bench = {}
for mode in ["linear", "log"]:
    m = LinearRegression().fit(dist_feats(Xtr[DIST_COL], mode), ytr)
    bench[mode] = (m, r2_score(yva, m.predict(dist_feats(Xva[DIST_COL], mode))))
    print(f"distance ({mode:6s}) validation R2 = {bench[mode][1]:.4f}")
best_mode = max(bench, key=lambda k: bench[k][1])
bench_model = bench[best_mode][0]
print("Benchmark uses:", best_mode, "distance")

bench_pred = {s: bench_model.predict(dist_feats(Xs[DIST_COL], best_mode))
              for s, Xs in [("train", Xtr), ("val", Xva), ("test", Xte)]}
for s, yy in [("train", ytr), ("val", yva), ("test", yte)]:
    results.append(metrics("Distance-only benchmark", s, yy, bench_pred[s]))
pd.DataFrame(results)

## 7. Ridge Regression
Ridge minimises  ‖y − Xw‖² + α‖w‖².  The L2 penalty shrinks coefficients, which stabilises the fit when features are correlated (here, several distance measures and PGA/PGV/PGD are strongly correlated – see Member 1's heatmap).

**Tuning α:** 5-fold *GroupKFold* cross-validation on the training set (folds never split an earthquake), scoring on R². The validation set is kept for model comparison and the test set is touched only once at the end.

In [ ]:
pre = ColumnTransformer([("scale", StandardScaler(), list(X.columns))])
ridge_pipe = Pipeline([("pre", pre), ("ridge", Ridge(random_state=SEED))])
alphas = np.logspace(-3, 4, 30)
gs = GridSearchCV(ridge_pipe, {"ridge__alpha": alphas}, cv=GroupKFold(n_splits=5),
                  scoring="r2", return_train_score=True, n_jobs=-1)
gs.fit(Xtr, ytr, groups=gtr)
best_alpha = gs.best_params_["ridge__alpha"]
print(f"Best alpha = {best_alpha:.4g}   (CV R2 = {gs.best_score_:.4f})")

In [ ]:
cv = pd.DataFrame(gs.cv_results_)
plt.semilogx(alphas, cv["mean_train_score"], label="train (CV)")
plt.semilogx(alphas, cv["mean_test_score"], label="validation (CV)")
plt.axvline(best_alpha, ls="--", c="k", label=f"best α = {best_alpha:.3g}")
plt.xlabel("α (regularisation strength)"); plt.ylabel("R²"); plt.legend(); plt.title("Ridge: α tuning")
plt.tight_layout(); plt.savefig("results/figures/m2_ridge_alpha_tuning.png", dpi=150); plt.show()

In [ ]:
ridge = gs.best_estimator_
ridge_pred = {s: ridge.predict(Xs) for s, Xs in [("train", Xtr), ("val", Xva), ("test", Xte)]}
for s, yy in [("train", ytr), ("val", yva), ("test", yte)]:
    results.append(metrics("Ridge Regression", s, yy, ridge_pred[s]))
res = pd.DataFrame(results)
res.round(4)

## 8. Results table (saved for Member 3's final comparison)

In [ ]:
res.round(5).to_csv("results/metrics_member2.csv", index=False)
json.dump({"best_alpha": float(best_alpha), "benchmark_distance_mode": best_mode,
           "n_features": int(X.shape[1]), "features": list(X.columns), "seed": SEED},
          open("results/ridge_config.json", "w"), indent=2)
joblib.dump(ridge, "results/ridge_model.joblib")
res.pivot(index="model", columns="split", values=["R2", "MAE"]).round(4)

## 9. Plots
### 9.1 True vs predicted (test set)
Points on the dashed line = perfect prediction. A flat horizontal cloud means the model is just predicting a near-constant.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for a, (nm, p) in zip(ax, [("Distance-only benchmark", bench_pred["test"]), ("Ridge Regression", ridge_pred["test"])]):
    a.scatter(yte, p, s=6, alpha=.3)
    lim = [0, max(yte.max(), p.max())]; a.plot(lim, lim, "k--")
    a.set_title(f"{nm}\nR² = {r2_score(yte, p):.3f}, MAE = {mean_absolute_error(yte, p):.2f}°")
    a.set_xlabel("True Δθ (deg)")
ax[0].set_ylabel("Predicted Δθ (deg)")
plt.tight_layout(); plt.savefig("results/figures/m2_true_vs_pred.png", dpi=150); plt.show()

### 9.2 Trend plot – Δθ vs site distance (binned means)

In [ ]:
t = pd.DataFrame({"d": Xte[DIST_COL].values, "true": yte, "bench": bench_pred["test"], "ridge": ridge_pred["test"]})
t["bin"] = pd.qcut(t["d"], q=15, duplicates="drop")
g = t.groupby("bin", observed=True).agg(d=("d", "median"), true=("true", "mean"), bench=("bench", "mean"), ridge=("ridge", "mean"))
plt.plot(g.d, g.true, "o-", label="True (bin mean)"); plt.plot(g.d, g.bench, "s--", label="Distance-only benchmark")
plt.plot(g.d, g.ridge, "^--", label="Ridge")
plt.xscale("log"); plt.xlabel("Site-to-site distance (km)"); plt.ylabel("Mean Δθ (deg)"); plt.legend()
plt.title("Does the model capture the trend of Δθ with distance?")
plt.tight_layout(); plt.savefig("results/figures/m2_trend_vs_distance.png", dpi=150); plt.show()

### 9.3 Residuals (Ridge, test set)

In [ ]:
resid = yte - ridge_pred["test"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(ridge_pred["test"], resid, s=6, alpha=.3); ax[0].axhline(0, c="k", ls="--")
ax[0].set_xlabel("Predicted Δθ"); ax[0].set_ylabel("Residual"); ax[0].set_title("Residuals vs predicted")
sns.histplot(resid, bins=40, kde=True, ax=ax[1]); ax[1].set_title("Residual distribution")
plt.tight_layout(); plt.savefig("results/figures/m2_ridge_residuals.png", dpi=150); plt.show()

### 9.4 Ridge coefficients (standardised features → directly comparable)

In [ ]:
coef = pd.Series(ridge.named_steps["ridge"].coef_, index=X.columns).sort_values(key=abs, ascending=False)
top = coef.head(15)[::-1]
top.plot.barh(color=np.where(top > 0, "#2980b9", "#c0392b"))
plt.xlabel("Coefficient (per 1 std of feature)"); plt.title("Top Ridge coefficients")
plt.tight_layout(); plt.savefig("results/figures/m2_ridge_coefficients.png", dpi=150); plt.show()
coef.head(10)

## 10. Interpretation (fill in after running on the real data)
* **Benchmark:** state its test R² / MAE and whether Δθ clearly changes with distance (see trend plot).
* **Ridge vs benchmark:** report the improvement in R² and MAE. Is it large or marginal?
* **Train vs test gap:** a small gap → regularisation is working; a large gap → overfitting.
* **Coefficients:** which features dominate? Do they make physical sense (distance, Vs30, magnitude, mechanism)?
* **Limitations:** Ridge is linear – it cannot represent non-linear interactions (e.g. distance × soil), which motivates Member 3's neural network. The target is an angle difference with a bounded range, and directionality is intrinsically noisy, so low R² values are expected and not necessarily a coding error.